In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df1 = pd.read_csv("/kaggle/input/datasets/hananfath/enron-data-fraud/enron_data_fraud_labeled.csv")

In [ ]:
print(df1.shape)

In [ ]:
df2 = pd.read_csv("/kaggle/input/datasets/hananfath/name-emp/enron_emp.csv")

In [ ]:
print(df2.shape)

In [ ]:
df2.head()

In [ ]:
print(df2.columns.tolist())

In [ ]:
df2 = pd.read_csv(
    "/kaggle/input/datasets/hananfath/name-emp/enron_emp.csv",
    sep="\t"
)

In [ ]:
print(df2.shape)

In [ ]:
print(df2.columns.tolist())

In [ ]:
df2["Unnamed: 5"].isnull().sum()

In [ ]:
df2[df2["Unnamed: 5"].notnull()]

In [ ]:
df2 = df2.rename(columns={"Unnamed: 5": "email4"})

In [ ]:
print(df2.columns.tolist())

In [ ]:
print(df1.columns.tolist())

In [ ]:
df1["From"] = df1["From"].astype(str)
df1["To"] = df1["To"].astype(str)
df1["Body"] = df1["Body"].astype(str)

In [ ]:
type(df1.loc[0, "From"])
type(df1.loc[0, "To"])
type(df1.loc[0, "Body"])

In [ ]:
columns_to_drop = [
    "Message-ID",
    "Mime-Version",
    "Content-Type",
    "Content-Transfer-Encoding",
    "Mail-ID",
    "Folder-User",
    "Subject",
    "X-cc",
    "X-bcc",
    "Time",
    "X-Origin",
    "Cc",
    "Bcc",
    "Attendees",
    "Re",
    "Source",
    "Suspicious-Folders"
]

df1 = df1.drop(columns=columns_to_drop)

In [ ]:
print(df1.shape)

In [ ]:
df1["Date"].head()

In [ ]:
df1["Date"] = pd.to_datetime(df1["Date"], utc=True)

df1["Date"] = df1["Date"].dt.strftime("%b-%Y")

In [ ]:
df1["Date"].head()

In [ ]:
df1["Year"] = df1["Date"].str[-4:]

In [ ]:
df1[["Date", "Year"]].head()

In [ ]:
year_counts = df1["Year"].value_counts().sort_index()

print(year_counts)

In [ ]:
import matplotlib.pyplot as plt

year_counts.plot(kind="bar")

plt.title("Number of Emails by Year")
plt.xlabel("Year")
plt.ylabel("Email Count")

plt.show()

In [ ]:
df1["Year"] = df1["Year"].astype(int)

df1 = df1[(df1["Year"] >= 1995) & (df1["Year"] <= 2003)]

In [ ]:
print(df1.shape)

print(df1["Year"].min())
print(df1["Year"].max())

In [ ]:
df1 = df1.dropna(subset=["From", "To"])

In [ ]:
print(df1.shape)

print(df1[["From", "To"]].isnull().sum())

In [ ]:
df1["X-Folder"].head()

In [ ]:
df1["X-Folder"] = df1["X-Folder"].str.split("\\").str[-1]

In [ ]:
df1["X-Folder"].head()

In [ ]:
df1[df1["Contains-Reply-Forwards"] == True]["Body"].head(10)

In [ ]:
import re

def remove_fwd_re(text):
    if pd.isnull(text):
        return ""

    # remove only prefixes at start
    text = re.sub(r'^\s*fwd:\s*', '', text, flags=re.IGNORECASE)
    text = re.sub(r'^\s*re:\s*', '', text, flags=re.IGNORECASE)

    return text

In [ ]:
df1["Body"] = df1["Body"].apply(remove_fwd_re)

In [ ]:
import re

def clean_reply_forward_tags(text):
    if pd.isnull(text):
        return ""

    # remove only Fwd:/RE: prefixes
    text = re.sub(r'^\s*fwd:\s*', '', text, flags=re.IGNORECASE)
    text = re.sub(r'^\s*re:\s*', '', text, flags=re.IGNORECASE)

    # remove ONLY header lines (not entire body)
    text = re.sub(r'^from:.*$', '', text, flags=re.IGNORECASE | re.MULTILINE)
    text = re.sub(r'^sent:.*$', '', text, flags=re.IGNORECASE | re.MULTILINE)
    text = re.sub(r'^to:.*$', '', text, flags=re.IGNORECASE | re.MULTILINE)
    text = re.sub(r'^subject:.*$', '', text, flags=re.IGNORECASE | re.MULTILINE)
    text = re.sub(r'^cc:.*$', '', text, flags=re.IGNORECASE | re.MULTILINE)

    # remove forwarded markers
    text = re.sub(r'forwarded by.*$', '', text, flags=re.IGNORECASE | re.MULTILINE)
    text = re.sub(r'begin forwarded message.*$', '', text, flags=re.IGNORECASE | re.MULTILINE)

    # remove quoted reply lines only
    text = re.sub(r'^>.*$', '', text, flags=re.MULTILINE)

    # clean extra spaces
    text = re.sub(r'\s+', ' ', text).strip()

    return text

In [ ]:
df1["Body"] = df1["Body"].apply(clean_reply_forward_tags)

In [ ]:
df1[df1["Contains-Reply-Forwards"] == True]["Body"].head(10)

In [ ]:
import nltk
import string
from nltk.corpus import stopwords
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))
stop_words.update([
    'mailto', 'pm', 'am', 'Forwarded', 'message','by',
    'from', 'subject', 're', 'edu', 'use', 'to',
    'cc', 'http', 'sent', 'enron', 'com','----------------------', 'enroncom'
])

def remove_urls(text):
    pattern = re.compile(r'https?://\S+|www\.\S+')
    return re.sub(pattern, "", str(text))

def remove_emails(text):
    pattern = re.compile(r"[\w\.-]+@[\w\.-]+\.\w+")
    return re.sub(pattern, "", str(text))

def remove_punctuations(text):
    return re.sub('[%s]' % re.escape(string.punctuation), " ", str(text))

def remove_digits(text):
    pattern = re.compile(r"\w*\d+\w*")
    return re.sub(pattern, "", str(text))

def remove_stopwords(text):
    return " ".join([word for word in str(text).split() if word not in stop_words])

def remove_extra_spaces(text):
    return re.sub(r'\s+', ' ', str(text)).strip()

In [ ]:
# =========================
# FULL TEXT PREPROCESSING PIPELINE (NO DIGIT REMOVAL)
# =========================

# Step 1: Lowercase + handle missing values
df1["text_lower"] = df1["Body"].fillna("").str.lower()

# Step 2: Remove URLs
df1["text_noURLs"] = df1["text_lower"].apply(remove_urls)

# Step 3: Remove Emails
df1["text_noEmails"] = df1["text_noURLs"].apply(remove_emails)

# Step 4: Remove Punctuation
df1["text_noPuncts"] = df1["text_noEmails"].apply(remove_punctuations)

# Step 5: Remove Stopwords
df1["text_noStopwords"] = df1["text_noPuncts"].apply(remove_stopwords)

# Step 6: Remove Extra Spaces
df1["text_noExtraspace"] = df1["text_noStopwords"].apply(remove_extra_spaces)

# Step 7: Final processed body column
df1["Processed_Body"] = df1["text_noExtraspace"]

In [ ]:
df1["Processed_Body"] = df1["text_noExtraspace"]

In [ ]:
!pip install spacy
!python -m spacy download en_core_web_sm

In [ ]:
import spacy

In [ ]:
nlp = spacy.load("en_core_web_sm")

In [ ]:
nlp.max_length = 2000000

In [ ]:
df1["Processed_Body"] = df1["Processed_Body"].astype(str).str[:1000000]

In [ ]:
processed_texts = []

for doc in nlp.pipe(df1["Processed_Body"].fillna(""), batch_size=500):
    tokens = [token.text for token in doc]
    processed_texts.append(" ".join(tokens))

In [ ]:
df1["From"] = df1["From"].fillna("").astype(str).str.strip().str.lower()
df1["To"] = df1["To"].fillna("").astype(str).str.strip().str.lower()

In [ ]:
df1 = df1.drop_duplicates(subset=["From", "To", "Body"])

In [ ]:
df1.shape

In [ ]:
df1["To"].dropna().head(10)

In [ ]:
# Step 1: standardize separators
df1["To"] = (
    df1["To"]
    .fillna("")
    .astype(str)
    .str.replace(";", ",")
    .str.replace("\n", ",")
)

# Step 2: split
df1["To"] = df1["To"].str.split(",")

# Step 3: explode
df1 = df1.explode("To")

# Step 4: strip both columns (mentor + fix)
df1["To"] = df1["To"].str.strip().str.lower()
df1["From"] = df1["From"].str.strip().str.lower()

In [ ]:
df1.shape

In [ ]:
df1["From"] = df1["From"].fillna("").astype(str).str.strip().str.lower()
df1["To"] = df1["To"].fillna("").astype(str).str.strip().str.lower()

In [ ]:
email_pattern = r'^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$'

df1 = df1[
    df1["From"].str.match(email_pattern, na=False) &
    df1["To"].str.match(email_pattern, na=False)
]

In [ ]:
df1.shape

In [ ]:
df1 = df1.drop_duplicates()

In [ ]:
df1 = df1.drop_duplicates(subset=["From", "To", "Body"])

In [ ]:
df1.shape

In [ ]:
df1 = df1[df1["Processed_Body"].notna()]

In [ ]:
df1 = df1[df1["Processed_Body"].str.strip() != ""]

In [ ]:
df1.shape

In [ ]:
# =========================
# STEP 1: EMAIL MAP (DF2)
# =========================
email_map = {}

for _, row in df2.iterrows():
    name = row["name"]
    for col in ["email1", "email2", "email3", "email4"]:
        email = row[col]
        if pd.notna(email):
            email_map[email.strip().lower()] = name

In [ ]:
# =========================
# STEP 2: PRIMARY MAPPING (HIGHEST PRIORITY)
# =========================
df1["From_Name"] = df1["From"].str.lower().map(email_map)
df1["To_Name"] = df1["To"].str.lower().map(email_map)

In [ ]:
# =========================
# STEP 3: X-FROM / X-TO CLEANING
# =========================
import re

def clean_name(x):
    if not isinstance(x, str) or not x:
        return ""
    
    x = re.sub(r"<.*?>", "", x)
    x = re.sub(r"/.*", "", x)
    x = re.sub(r"@[a-zA-Z0-9.-]+", "", x)
    
    return x.strip()

In [ ]:
# =========================
# STEP 4: HANDLE X FIELDS
# =========================
df1["X-From"] = df1["X-From"].fillna("")
df1["X-To"] = df1["X-To"].fillna("")

In [ ]:
# =========================
# STEP 5: EXTRACT X-NAMES (SECOND PRIORITY)
# =========================
df1["From_Name_X"] = df1["X-From"].apply(clean_name)
df1["To_Name_X"] = df1["X-To"].apply(clean_name)

In [ ]:
# =========================
# STEP 6: FILL ONLY NULL VALUES (NO OVERWRITE)
# =========================
df1["From_Name"] = df1["From_Name"].fillna(df1["From_Name_X"])
df1["To_Name"] = df1["To_Name"].fillna(df1["To_Name_X"])

In [ ]:
# =========================
# STEP 7: FINAL FALLBACK (LAST PRIORITY)
# =========================
df1["From_Name"] = df1["From_Name"].fillna(df1["From"].str.split("@").str[0])
df1["To_Name"] = df1["To_Name"].fillna(df1["To"].str.split("@").str[0])

In [ ]:
# =========================
# STEP 8: NORMALIZATION
# =========================
df1["From_Name"] = df1["From_Name"].str.strip().str.lower()
df1["To_Name"] = df1["To_Name"].str.strip().str.lower()

In [ ]:
# =========================
# STEP 9: REMOVE NULLS 
# =========================
df1 = df1.dropna(subset=["From_Name", "To_Name"])

In [ ]:
# =========================
# STEP 10: FINAL DEDUPLICATION
# =========================
df1 = df1.drop_duplicates(
    subset=["From_Name", "To_Name", "Processed_Body"]
)

In [ ]:
df1.shape

In [ ]:
# Check current label distribution
print(df1['Label'].value_counts())
print(df1['Label'].value_counts(normalize=True))

In [ ]:
df_full_preprocessed = df1.copy()

In [ ]:
from sklearn.model_selection import train_test_split

df_reduced, _ = train_test_split(
    df1,
    train_size=230000,
    stratify=df1['Label'],
    random_state=42
)

print(df_reduced.shape)
print(df_reduced['Label'].value_counts())

In [ ]:
df1 = df_reduced.copy()

print(df1.shape)
print(df1['Label'].value_counts())

In [ ]:
df1 = df_reduced.copy()

print(df1.shape)

In [ ]:
df1["From_Name"] = df1["From"].str.lower().map(email_map)
df1["To_Name"] = df1["To"].str.lower().map(email_map)

In [ ]:
print(df1["From_Name"].isna().sum())
print(df1["To_Name"].isna().sum())

In [ ]:
df1["X-From"] = df1["X-From"].fillna("")

df1["From_Name_X"] = df1["X-From"].apply(clean_name)

df1["From_Name"] = df1["From_Name"].fillna(df1["From_Name_X"])

In [ ]:
print(df1["From_Name"].isna().sum())
print(df1["To_Name"].isna().sum())

In [ ]:
df1["To_Name"] = df1["To_Name"].fillna(
    df1["To"].str.split("@").str[0]
)

In [ ]:
print(df1["From_Name"].isna().sum())
print(df1["To_Name"].isna().sum())

In [ ]:
df1["From_Name"] = df1["From_Name"].astype(str).str.strip().str.lower()
df1["To_Name"] = df1["To_Name"].astype(str).str.strip().str.lower()

In [ ]:
print(df1["To_Name"].str.contains(",", na=False).sum())

In [ ]:
df1 = df1.drop_duplicates(
    subset=["From_Name", "To_Name", "Processed_Body"]
)

print(df1.shape)

In [ ]:
import networkx as nx

In [ ]:
print((df1["From_Name"] == "").sum())
print((df1["To_Name"] == "").sum())

In [ ]:
# Treat blank From_Name as missing
df1["From_Name"] = df1["From_Name"].replace("", pd.NA)

# Method 1: Email mapping
mapped_from = df1["From"].str.lower().map(email_map)
df1["From_Name"] = df1["From_Name"].fillna(mapped_from)

# Method 2: X-From fallback (only remaining missing values)
xfrom_names = df1["X-From"].fillna("").apply(clean_name)
df1["From_Name"] = df1["From_Name"].fillna(xfrom_names)

# If X-From produced blanks, convert them back to missing
df1["From_Name"] = df1["From_Name"].replace("", pd.NA)

# Method 3: Email prefix fallback (only remaining missing values)
prefix_from = df1["From"].str.split("@").str[0]
df1["From_Name"] = df1["From_Name"].fillna(prefix_from)

# FINAL STANDARDIZATION
df1["From_Name"] = (
    df1["From_Name"]
    .astype(str)
    .str.strip()
    .str.lower()
)

In [ ]:
print("Blank From_Name:", (df1["From_Name"].str.strip() == "").sum())
print("Null From_Name:", df1["From_Name"].isna().sum())

In [ ]:
df1 = df1.drop_duplicates(
    subset=["From_Name", "To_Name", "Processed_Body"]
)

print(df1.shape)

In [ ]:
df_full_preprocessed.shape

In [ ]:
df_full_preprocessed.to_csv(
    "enron_full_preprocessed.csv",
    index=False
)

In [ ]:
df1.to_csv(
    "enron_sampled_dataset.csv",
    index=False
)

In [ ]:
node_features.to_csv(
    "node_features.csv",
    index=False
)

In [ ]:
import os

print(os.listdir())

In [ ]:
import os

for file in [
    "enron_full_preprocessed.csv",
    "enron_sampled_dataset.csv",
    "node_features.csv"
]:
    size_mb = os.path.getsize(file) / (1024 * 1024)
    print(f"{file}: {size_mb:.2f} MB")

In [ ]:
pd.DataFrame({
    "Full_Dataset_Columns": pd.Series(df_full_preprocessed.columns),
    "Sampled_Dataset_Columns": pd.Series(df1.columns)
})

In [ ]:
cols_to_drop = [
    "text_lower",
    "text_noURLs",
    "text_noEmails",
    "text_noPuncts",
    "text_noStopwords",
    "text_noExtraspace"
]

df_full_preprocessed = df_full_preprocessed.drop(columns=cols_to_drop)

df1 = df1.drop(columns=cols_to_drop)

print(df_full_preprocessed.shape)
print(df1.shape)

In [ ]:
df_full_preprocessed.to_parquet(
    "enron_full_preprocessed_v2.parquet",
    index=False
)

df1.to_parquet(
    "enron_sampled_dataset_v2.parquet",
    index=False
)

In [ ]:
import os

files = [
    "enron_full_preprocessed_v2.parquet",
    "enron_sampled_dataset_v2.parquet"
]

for file in files:
    size_mb = os.path.getsize(file) / (1024 * 1024)
    print(f"{file}: {size_mb:.2f} MB")

In [ ]:
import pandas as pd

test_full = pd.read_parquet("enron_full_preprocessed_v2.parquet")
test_sampled = pd.read_parquet("enron_sampled_dataset_v2.parquet")

print(test_full.shape)
print(test_sampled.shape)

In [ ]:
import os

for f in os.listdir():
    if f.endswith(".parquet"):
        print(f)

In [ ]:
import zipfile

files_to_zip = [
    "enron_full_preprocessed_v2.parquet",
    "enron_sampled_dataset_v2.parquet"
]

for file in files_to_zip:
    zip_name = file.replace(".parquet", ".zip")

    with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as zipf:
        zipf.write(file)

    print(f"Created: {zip_name}")

In [ ]:
import os

[f for f in os.listdir() if f.endswith(".zip")]

In [ ]:
from IPython.display import FileLink

FileLink("enron_full_preprocessed_v2.parquet")

In [ ]:
FileLink("enron_sampled_dataset_v2.parquet")